# tsauditor — production-hardening walkthrough

This notebook demonstrates four additions that take tsauditor from a
leakage detector toward a production data guard:

1. **LEK004 — as-of / point-in-time leakage**: a value used before it was
   published (macro releases, sentiment, earnings).
2. **Validity checks (VAL001/VAL002)**: values that are *definitionally*
   wrong — a non-positive spread, a sentiment score outside [-1, 1], a
   crossed order book.
3. **`tsa.fix()`**: one-shot scan-and-repair returning `(clean_df, report)`,
   so the audit trail is never discarded and the original is never touched.
4. **`tsa.adapters.to_timesfm()`**: audit + repair + format a series into the
   finite float32 array Google TimesFM expects.

Everything here runs with just `tsauditor` installed — no `timesfm` needed.

In [1]:
import numpy as np
import pandas as pd
import tsauditor as tsa

print("tsauditor", tsa.__version__)

tsauditor 0.2.0


## A deliberately messy market frame

We build a small daily frame with one problem per feature so each check has
something to find:

* `price` — a clustered missing gap and a single fat outlier
* `cpi` — a macro series aligned to its *reference* date but published ~30
  days later (the as-of trap)
* `bid` / `ask` — mostly well-ordered, with a couple of crossed-book rows
* `sentiment` — meant to live in [-1, 1], with a couple of out-of-range values

In [2]:
idx = pd.date_range("2021-01-01", periods=300, freq="B")
rng = np.random.default_rng(7)

price = 100 + np.cumsum(rng.normal(0, 1, 300))
price[120] = price[120] + 60  # fat outlier
price[40:50] = np.nan  # clustered gap

cpi = np.repeat(np.linspace(2.0, 5.0, 30), 10)  # stepwise macro series

bid = 100 + rng.normal(0, 0.5, 300)
ask = bid + rng.uniform(0.02, 0.10, 300)  # ask above bid...
ask[[75, 210]] = bid[[75, 210]] - 0.05  # ...except two crossed rows

sentiment = rng.uniform(-0.8, 0.8, 300)
sentiment[[15, 250]] = [1.7, -2.1]  # out of [-1, 1]

df = pd.DataFrame(
    {"price": price, "cpi": cpi, "bid": bid, "ask": ask, "sentiment": sentiment},
    index=idx,
)
df.head()

,price,cpi,bid,ask,sentiment
2021-01-01,100.001230,2.0,100.755915,100.844194,0.458481
2021-01-04,100.299976,2.0,100.277844,100.330104,-0.076153
2021-01-05,100.025838,2.0,99.970770,99.991032,-0.516711
2021-01-06,99.135246,2.0,99.710304,99.793261,-0.743516
2021-01-07,98.680575,2.0,99.682501,99.722139,0.562207


## 1. As-of leakage (LEK004)

CPI for a reference month is only released a few weeks later. If the series is
aligned to the reference date and used on that date, every row before the real
release consumes future information. tsauditor cannot know publication dates on
its own, so you declare them — here as a fixed 30-day publication lag.

A `pd.Timedelta` says *the value at each row became available 30 days later*,
so an unshifted column is used early on every row. (For a real ragged release
calendar, pass a `pd.Series` of per-row publish timestamps instead.)

In [3]:
report = tsa.scan(
    df,
    available_at={"cpi": pd.Timedelta(days=30)},
    run_stationarity=False,
)

for i in report.filter(code="LEK004"):
    print(i.code, "-", i.column)
    print("  ", i.description)
    print("  evidence:", i.evidence)

LEK004 - cpi
   Feature 'cpi' is used before it was available: 300 row(s) carry a value whose release time is later than the row's own timestamp (max look-ahead 30.0 days, first at 2021-01-01 00:00:00). Rows before each release consume future information — align the column to its release schedule.
  evidence: {'n_violations': 300, 'max_lookahead_days': 30.0, 'first_violation': '2021-01-01 00:00:00', 'check': 'as-of'}


The fix is **not** to drop `cpi` — it is to shift it to its release schedule so
each value is only used on or after it was published. The suggestion says so:

In [4]:
next(s for s in report.suggestions() if s["code"] == "LEK004")

{'code': 'LEK004',
 'column': 'cpi',
 'severity': 'critical',
 'suggestion': "Align column 'cpi' to its release schedule: some values sit at a timestamp earlier than when they were published, so rows before each release use future information. Shift the column forward to its availability dates (not reference dates) rather than dropping it."}

## 2. Domain-validity checks (VAL001 / VAL002)

These catch values that are impossible rather than merely surprising. You declare
the rules; tsauditor verifies them.

* `bounds` — per-column limits (sentiment must be within [-1, 1])
* `relations` — ordered `(low, high)` pairs; `('bid', 'ask')` must hold every row,
  so a crossed book (ask < bid) is flagged CRITICAL.

In [5]:
report = tsa.scan(
    df,
    constraints={
        "bounds": {"sentiment": {"min": -1, "max": 1}},
        "relations": [("bid", "ask")],
    },
    run_stationarity=False,
)

for i in report.filter(module="validity"):
    print(i.severity.upper(), i.code, "-", i.column, "| n=", i.evidence["n_violations"])

# Validity issues are data errors, not leakage:
print("leaky_columns:", report.leaky_columns())

CRITICAL VAL002 - ask | n= 2
WARNING VAL001 - sentiment | n= 2
leaky_columns: []


## 3. One-shot repair with `tsa.fix()`

`tsa.fix()` scans and repairs in a single call and returns **both** the cleaned
copy and the report, so you keep the audit trail. The original frame is never
modified — it is your backup.

In [6]:
clean, fix_report = tsa.fix(df, domain="finance")

print("original still has the gap? ", df["price"].isna().sum(), "NaNs")
print("cleaned gap filled?       ", clean["price"].isna().sum(), "NaNs")
print("original is untouched:    ", not clean.equals(df))
print()
print("change log (report.last_fixes):")
for entry in fix_report.last_fixes:
    print(
        "  ",
        entry["column"],
        "->",
        entry["action"],
        "(",
        entry["cells_changed"],
        "cells )",
    )

original still has the gap?  10 NaNs
cleaned gap filled?        0 NaNs
original is untouched:     True

change log (report.last_fixes):
   price -> clip_outliers ( 1 cells )
   bid -> clip_outliers ( 2 cells )
   ask -> clip_outliers ( 2 cells )
   sentiment -> clip_outliers ( 1 cells )
   price -> clip_spikes ( 1 cells )
   cpi -> stuck_to_nan ( 300 cells )
   price -> impute_interpolate ( 10 cells )


Because imputed values are *estimates*, the log matters: it tells you exactly
which cells were fabricated so you can decide whether to trust them downstream.

## 4. The TimesFM bridge: `tsa.adapters.to_timesfm()`

Zero-shot forecasters like Google TimesFM tokenize a clean, contiguous, finite
context window; a raw series with gaps makes tokenization fail. The adapter
audits, repairs, and returns a plain `float32` array — and **verifies it is
finite** before returning, so a NaN never reaches the model.

Note it cleans the target series as an ordinary column (it is the thing you
forecast), unlike `fix(target=...)` which protects a label.

In [7]:
array, prep_report = tsa.adapters.to_timesfm(
    df, target_col="price", domain="finance", context_len=1024, return_report=True
)

print("dtype      :", array.dtype)
print("shape      :", array.shape)
print("all finite :", bool(np.isfinite(array).all()))
print("repaired   :", [e["column"] for e in prep_report.last_fixes])

dtype      :

 float32
shape      : (300,)
all finite : True
repaired   : ['price', 'bid', 'ask', 'sentiment', 'price', 'cpi', 'price']


### Handing the array to TimesFM

The adapter adds no `timesfm` dependency, so the model call lives in your code.
With `pip install timesfm[torch]` it looks like this (left un-run here):

```python
import timesfm

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    'google/timesfm-2.5-200m-pytorch'
)
model.compile(timesfm.ForecastConfig(max_context=1024, max_horizon=64))

point_forecast, quantiles = model.forecast(horizon=24, inputs=[array])
```

`context_len` / `min_context` in the adapter are your knobs, not TimesFM
constants — TimesFM 2.5 accepts a wide range of context lengths (up to 16k).
Verify the settings for the model version you target.

## Recap

| Concern | Entry point | Code |
|---|---|---|
| Value used before release | `scan(available_at=...)` | LEK004 |
| Impossible / out-of-range values | `scan(constraints=...)` | VAL001 / VAL002 |
| One-shot repair + audit trail | `tsa.fix(df)` | — |
| Forecast-ready array | `tsa.adapters.to_timesfm(df, col)` | — |

All checks are opt-in and declarative: tsauditor never guesses release dates or
validity rules, and never edits your data unless you ask it to.